In [ ]:
# ======================
# IMPORTS
# ======================
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing import image

print("TensorFlow:", tf.__version__)

# ======================
# CONFIG
# ======================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

BASE_DIR = os.getcwd()   # IMPORTANT
DATA_DIR = os.path.join(BASE_DIR, "dataset")
MODEL_DIR = os.path.join(BASE_DIR, "model")

os.makedirs(MODEL_DIR, exist_ok=True)

print("Dataset path:", DATA_DIR)
print("Exists:", os.path.exists(DATA_DIR))

# ======================
# DATA GENERATOR
# ======================
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print("Classes:", train_gen.class_indices)

# ======================
# SAVE LABELS
# ======================
labels_path = os.path.join(MODEL_DIR, "cabbage_labels.json")

class_labels = {v: k for k, v in train_gen.class_indices.items()}

with open(labels_path, "w") as f:
    json.dump(class_labels, f, indent=4)

# ======================
# MODEL
# ======================
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ======================
# CALLBACKS
# ======================
checkpoint_path = os.path.join(MODEL_DIR, "cabbage_model.keras")

callbacks_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# ======================
# TRAINING
# ======================
history = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks_list
)

# ======================
# FINE TUNING
# ======================
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=callbacks_list
)

print("Training complete!")

# ======================
# PREDICTION FUNCTION
# ======================
def predict_disease(img_path):
    model = tf.keras.models.load_model(checkpoint_path)

    with open(labels_path, "r") as f:
        labels = json.load(f)

    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)[0]
    class_idx = np.argmax(preds)
    confidence = float(preds[class_idx])

    return {
        "disease": labels[str(class_idx)],
        "confidence": round(confidence * 100, 2)
    }

# Example:
# print(predict_disease("test.jpg"))

TensorFlow: 2.21.0
Dataset path: F:\krishi nova project ml part\cabbage\dataset
Exists: True
Found 1280 images belonging to 8 classes.
Found 320 images belonging to 8 classes.
Classes: {'Alternaria_Leaf_Spot': 0, 'Bacterial spot rot': 1, 'Black Rot': 2, 'Cabbage aphid colony': 3, 'Downy Mildew': 4, 'No disease': 5, 'club root': 6, 'ring spot': 7}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)          │ (None, 7, 7, 1280)          │       4,049,571 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 1280)                │           5,120 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 8)                   │           2,056 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,384,683 (16.73 MB)

 Trainable params: 332,552 (1.27 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

Epoch 1/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.1136 - loss: 2.6409
Epoch 1: val_accuracy improved from None to 0.12500, saving model to F:\krishi nova project ml part\cabbage\model\cabbage_model.keras

Epoch 1: finished saving model to F:\krishi nova project ml part\cabbage\model\cabbage_model.keras
40/40 ━━━━━━━━━━━━━━━━━━━━ 461s 11s/step - accuracy: 0.1172 - loss: 2.6807 - val_accuracy: 0.1250 - val_loss: 2.0937
Epoch 2/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.1273 - loss: 2.5878
Epoch 2: val_accuracy did not improve from 0.12500
40/40 ━━━━━━━━━━━━━━━━━━━━ 152s 4s/step - accuracy: 0.1414 - loss: 2.5266 - val_accuracy: 0.1250 - val_loss: 2.0969
Epoch 3/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1724 - loss: 2.3831
Epoch 3: val_accuracy improved from 0.12500 to 0.14687, saving model to F:\krishi nova project ml part\cabbage\model\cabbage_model.keras

Epoch 3: finished saving model to F:\krishi nova project ml part\cabbage\model\cabbage_model.ke